In [11]:
import numpy as np
from collections import defaultdict

data = [
    ['m', 'b', 'true'],
    ['m', 's', 'true'],
    ['g', 'q', 'true'],
    ['h', 's', 'true'],
    ['g', 'q', 'true'],
    ['g', 'q', 'false'],
    ['g', 's', 'false'],
    ['h', 'b', 'false'],
    ['h', 'q', 'false'],
    ['m', 'b', 'false']
]

def calculate_probabilities(dataset):
    priors = defaultdict(float)
    likelihoods = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    total_samples = len(dataset)

    for a, b, cls in dataset:
        priors[cls] += 1
        likelihoods[cls]['A'][a] += 1
        likelihoods[cls]['B'][b] += 1

    for cls in priors:
        priors[cls] /= total_samples
        for attr in likelihoods[cls]:
            total = sum(likelihoods[cls][attr].values())
            for key in likelihoods[cls][attr]:
                likelihoods[cls][attr][key] /= total

    return priors, likelihoods

def predict(test_sample, priors, likelihoods):
    scores = {}
    for cls in priors:
        score = priors[cls]
        for i, feature in enumerate(['A', 'B']):
            prob = likelihoods[cls][feature].get(test_sample[i], 0)
            score *= prob
        scores[cls] = score
    return max(scores, key=scores.get)


priors, likelihoods = calculate_probabilities(data)


print("Priors:")
for cls, prob in priors.items():
    print(f"  P({cls}) = {prob:.4f}")

print("\nLikelihoods:")
for cls in likelihoods:
    print(f"  For Class = {cls}:")
    for attr, values in likelihoods[cls].items():
        print(f"    {attr}: {dict(values)}")

# Test sample
test_sample = ['m', 'q']
print("\nTesting Sample:", test_sample)


print("\nPrediction Calculations:")
for cls in ['true', 'false']:
    print(f"\nFor Class = {cls}:")
    pa = likelihoods[cls]['A'].get(test_sample[0], 0)
    pb = likelihoods[cls]['B'].get(test_sample[1], 0)
    print(f"P(A={test_sample[0]}|{cls}) = {pa:.4f}")
    print(f"P(B={test_sample[1]}|{cls}) = {pb:.4f}")
    print(f"P({cls}|X) (unnormalized) = {priors[cls]*pa*pb:.4f}")

# Get final prediction
predicted_class = predict(test_sample, priors, likelihoods)
print("\nPredicted Class for", test_sample, ":", predicted_class)

Priors:
  P(true) = 0.5000
  P(false) = 0.5000

Likelihoods:
  For Class = true:
    A: {'m': 0.4, 'g': 0.4, 'h': 0.2}
    B: {'b': 0.2, 's': 0.4, 'q': 0.4}
  For Class = false:
    A: {'g': 0.4, 'h': 0.4, 'm': 0.2}
    B: {'q': 0.4, 's': 0.2, 'b': 0.4}

Testing Sample: ['m', 'q']

Prediction Calculations:

For Class = true:
P(A=m|true) = 0.4000
P(B=q|true) = 0.4000
P(true|X) (unnormalized) = 0.0800

For Class = false:
P(A=m|false) = 0.2000
P(B=q|false) = 0.4000
P(false|X) (unnormalized) = 0.0400

Predicted Class for ['m', 'q'] : true
